# Week 1 Day 4 — Model Tuning, Regularization & Reproducible Pipelines

**Dataset:** UCI Adult / Census Income  
**Target:** predict whether income is `>50K` (`1`) vs `<=50K` (`0`)  
**Primary metric:** Precision (business goal: targeted high-income outreach)  
**Secondary metrics:** Recall, F1, ROC-AUC, Average Precision, Brier score

This notebook completes the Day 4 requirements: reproducible pipelines, controlled hyperparameter search, learning-curve diagnostics, regularization analysis, probability calibration, threshold selection, untouched test evaluation, and artifact saving.

**Re-run:** place `adult_dataset.csv` beside this notebook and use **Run All**. All random seeds are fixed at 42.

In [ ]:
# Reproducibility and imports
import os, sys, platform, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import __version__ as sklearn_version
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, brier_score_loss, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, precision_recall_curve
)
from scipy.stats import loguniform, randint

RANDOM_STATE = 42
N_ITER = 50                
CV_SPLITS = 5
warnings.filterwarnings("ignore")

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn_version)
print("Random state:", RANDOM_STATE)


## Task 1 — Fully Reproducible Pipeline

The preprocessing is fitted **inside** each pipeline, so imputation, scaling and encoding cannot leak information from validation/test data. Missing categorical values are imputed rather than dropped. All model randomness is controlled with `random_state=42`.

In [ ]:
# Load raw Adult data. The supplied CSV has no header.
COLS = ["age","workclass","fnlwgt","education","education-num","marital-status",
        "occupation","relationship","race","sex","capital-gain","capital-loss",
        "hours-per-week","native-country","income"]

DATA_PATH = "adult_dataset.csv"
if not os.path.exists(DATA_PATH):
    # Also support the common original filename if the user renames the supplied data.
    DATA_PATH = "adult-all.csv"

df = pd.read_csv(DATA_PATH, header=None, names=COLS)
df = df.apply(lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x))
df = df.replace("?", np.nan)
df["target"] = (df["income"].str.replace(".", "", regex=False) == ">50K").astype(int)
df = df.drop(columns="income")

print("Shape:", df.shape)
print("Positive class (>50K):", f"{df.target.mean():.2%}")
print(df.isna().sum().sort_values(ascending=False).head(5))


In [ ]:
# Fixed stratified 70/10/20 split: train / development / untouched test
X = df.drop(columns="target")
y = df["target"]

X_train_dev, X_test, y_train_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_dev, y_train_dev, test_size=0.125, stratify=y_train_dev, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "Dev:", X_dev.shape, "Test:", X_test.shape)


In [ ]:
num_cols = ["age","fnlwgt","education-num","capital-gain","capital-loss","hours-per-week"]
cat_cols = ["workclass","education","marital-status","occupation","relationship","race","sex","native-country"]

onehot_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

ordinal_preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ]), cat_cols)
])

cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)


## Task 2 — Hyperparameter Search

The searches optimize **precision**, use `StratifiedKFold`, `n_jobs=-1`, and fixed random state. `N_ITER=10` keeps the notebook practical for a one-hour submission; set it to 50 for the assignment's larger recommended budget if time permits.

In [ ]:
# Logistic Regression — regularization penalty and C
lr_pipe = Pipeline([
    ("pre", onehot_preprocessor),
    ("model", LogisticRegression(solver="liblinear", max_iter=2000, random_state=RANDOM_STATE))
])
lr_search = RandomizedSearchCV(
    lr_pipe,
    {"model__penalty": ["l1","l2"], "model__C": loguniform(0.01, 10)},
    n_iter=N_ITER, scoring="precision", cv=cv, n_jobs=-1,
    random_state=RANDOM_STATE, return_train_score=True
)
lr_search.fit(X_train, y_train)
print("LR best precision:", round(lr_search.best_score_, 4))
print("LR best params:", lr_search.best_params_)


In [ ]:
# Random Forest — tree complexity and ensemble size
rf_pipe = Pipeline([
    ("pre", onehot_preprocessor),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"))
])
rf_search = RandomizedSearchCV(
    rf_pipe,
    {
        "model__n_estimators": randint(80, 220),
        "model__max_depth": [None, 8, 12, 16, 20],
        "model__min_samples_leaf": randint(1, 8),
        "model__max_features": ["sqrt","log2",0.5]
    },
    n_iter=N_ITER, scoring="precision", cv=cv, n_jobs=-1,
    random_state=RANDOM_STATE, return_train_score=True
)
rf_search.fit(X_train, y_train)
print("RF best precision:", round(rf_search.best_score_, 4))
print("RF best params:", rf_search.best_params_)


In [ ]:
# HistGradientBoosting — learning rate, iterations, tree complexity and L2
hgb_pipe = Pipeline([
    ("pre", ordinal_preprocessor),
    ("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE))
])
hgb_search = RandomizedSearchCV(
    hgb_pipe,
    {
        "model__learning_rate": [0.03, 0.05, 0.08, 0.10],
        "model__max_iter": [100, 150, 200],
        "model__max_leaf_nodes": [15, 31, 63],
        "model__l2_regularization": [0, 0.1, 1, 5]
    },
    n_iter=N_ITER, scoring="precision", cv=cv, n_jobs=-1,
    random_state=RANDOM_STATE, return_train_score=True
)
hgb_search.fit(X_train, y_train)
print("HGB best precision:", round(hgb_search.best_score_, 4))
print("HGB best params:", hgb_search.best_params_)


In [ ]:
# Compare tuned candidates on the development set (test remains untouched)
searches = {"Logistic Regression": lr_search, "Random Forest": rf_search, "HistGradientBoosting": hgb_search}
comparison = []
for name, s in searches.items():
    p = s.best_estimator_.predict_proba(X_dev)[:,1]
    pred = (p >= 0.5).astype(int)
    comparison.append({
        "model": name,
        "precision": precision_score(y_dev,pred),
        "recall": recall_score(y_dev,pred),
        "f1": f1_score(y_dev,pred),
        "roc_auc": roc_auc_score(y_dev,p),
        "avg_precision": average_precision_score(y_dev,p)
    })
comparison = pd.DataFrame(comparison).sort_values("precision", ascending=False)
display(comparison)


## Task 3 — Diagnose Overfitting / Underfitting

A learning curve compares training and validation precision as the training set grows. A large persistent gap indicates variance/overfitting; both curves being low indicates bias/underfitting. For tree models, reducing depth/leaf complexity or increasing regularization can reduce variance.

In [ ]:
# Learning curve for the tuned HGB candidate
best_hgb = hgb_search.best_estimator_
sizes, train_scores, val_scores = learning_curve(
    best_hgb, X_train, y_train, cv=cv, scoring="precision",
    train_sizes=np.linspace(0.2, 1.0, 5), n_jobs=-1
)
plt.figure(figsize=(7,5))
plt.plot(sizes, train_scores.mean(axis=1), marker="o", label="Train precision")
plt.plot(sizes, val_scores.mean(axis=1), marker="o", label="Validation precision")
plt.xlabel("Training examples"); plt.ylabel("Precision")
plt.title("Learning Curve — Tuned HistGradientBoosting")
plt.legend(); plt.tight_layout()
plt.savefig("day4_learning_curve.png", dpi=140)
plt.show()


In [ ]:
# Logistic Regression: effect of regularization strength C
C_values = np.logspace(-3, 1, 9)
lr_train, lr_dev = [], []
for C in C_values:
    pipe = Pipeline([
        ("pre", onehot_preprocessor),
        ("model", LogisticRegression(C=C, penalty="l2", solver="liblinear",
                                     max_iter=2000, random_state=RANDOM_STATE))
    ])
    pipe.fit(X_train, y_train)
    lr_train.append(precision_score(y_train, pipe.predict(X_train)))
    lr_dev.append(precision_score(y_dev, pipe.predict(X_dev)))

plt.figure(figsize=(7,5))
plt.semilogx(C_values, lr_train, marker="o", label="Train")
plt.semilogx(C_values, lr_dev, marker="o", label="Validation")
plt.xlabel("C (inverse regularization strength)"); plt.ylabel("Precision")
plt.title("Logistic Regression — Regularization Effect")
plt.legend(); plt.tight_layout()
plt.savefig("day4_logistic_C_curve.png", dpi=140)
plt.show()


In [ ]:
# Tree complexity diagnostic: max_depth for a compact RF
depths = [4, 8, 12, 16, 20]
rf_train, rf_dev = [], []
for depth in depths:
    pipe = Pipeline([
        ("pre", onehot_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=120, max_depth=depth, min_samples_leaf=3,
            max_features="sqrt", class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1))
    ])
    pipe.fit(X_train, y_train)
    rf_train.append(precision_score(y_train, pipe.predict(X_train)))
    rf_dev.append(precision_score(y_dev, pipe.predict(X_dev)))

plt.figure(figsize=(7,5))
plt.plot(depths, rf_train, marker="o", label="Train")
plt.plot(depths, rf_dev, marker="o", label="Validation")
plt.xlabel("max_depth"); plt.ylabel("Precision")
plt.title("Random Forest — Tree Depth Diagnostic")
plt.legend(); plt.tight_layout()
plt.savefig("day4_rf_depth_curve.png", dpi=140)
plt.show()


## Task 4 — Probability Calibration & Threshold Selection

Calibration uses **sigmoid (Platt) calibration**. The development set is used to choose the operating threshold for maximum F1; the test set is not used for threshold selection. Thresholds are a business decision: increasing the threshold generally improves precision while reducing recall.

In [ ]:
# Calibrate the selected HGB pipeline using CV on training data
calibrated_model = CalibratedClassifierCV(
    best_hgb, method="sigmoid", cv=3, n_jobs=-1
)
calibrated_model.fit(X_train, y_train)

p_dev = calibrated_model.predict_proba(X_dev)[:,1]
p_test = calibrated_model.predict_proba(X_test)[:,1]

brier = brier_score_loss(y_test, p_test)
print("Test Brier score:", round(brier, 4))


In [ ]:
# Select threshold on development data to maximize F1
thresholds = np.arange(0.20, 0.71, 0.02)
threshold_rows = []
for t in thresholds:
    pred = (p_dev >= t).astype(int)
    threshold_rows.append({
        "threshold": t,
        "precision": precision_score(y_dev,pred,zero_division=0),
        "recall": recall_score(y_dev,pred,zero_division=0),
        "f1": f1_score(y_dev,pred,zero_division=0),
        "positive_rate": pred.mean()
    })
threshold_table = pd.DataFrame(threshold_rows)
best_threshold = float(threshold_table.loc[threshold_table["f1"].idxmax(), "threshold"])
display(threshold_table.sort_values("f1", ascending=False).head(10))
print("Selected threshold:", best_threshold)


In [ ]:
# Calibration plot
prob_true, prob_pred = calibration_curve(y_test, p_test, n_bins=10, strategy="quantile")
plt.figure(figsize=(6,5))
plt.plot(prob_pred, prob_true, marker="o", label="Calibrated HGB")
plt.plot([0,1],[0,1],"--", label="Perfect calibration")
plt.xlabel("Mean predicted probability"); plt.ylabel("Fraction positive")
plt.title("Calibration Plot — Final Model")
plt.legend(); plt.tight_layout()
plt.savefig("day4_calibration_curve.png", dpi=140)
plt.show()

# Threshold trade-off
plt.figure(figsize=(7,5))
plt.plot(threshold_table.threshold, threshold_table.precision, label="Precision")
plt.plot(threshold_table.threshold, threshold_table.recall, label="Recall")
plt.plot(threshold_table.threshold, threshold_table.f1, label="F1")
plt.axvline(best_threshold, linestyle="--", label=f"Chosen = {best_threshold:.2f}")
plt.xlabel("Classification threshold"); plt.ylabel("Score")
plt.title("Threshold Trade-off — Development Set")
plt.legend(); plt.tight_layout()
plt.savefig("day4_threshold_tradeoff.png", dpi=140)
plt.show()


In [ ]:
# Show how thresholds change confusion matrix and business KPIs
selected_thresholds = [0.30, best_threshold, 0.55]
kpi_rows=[]
fig, axes = plt.subplots(1,3,figsize=(14,4))
for ax,t in zip(axes, selected_thresholds):
    pred=(p_dev>=t).astype(int)
    cm=confusion_matrix(y_dev,pred)
    ConfusionMatrixDisplay(cm,display_labels=["<=50K",">50K"]).plot(ax=ax,colorbar=False)
    ax.set_title(f"Threshold {t:.2f}")
    kpi_rows.append({
        "threshold":t,
        "precision":precision_score(y_dev,pred,zero_division=0),
        "recall":recall_score(y_dev,pred,zero_division=0),
        "f1":f1_score(y_dev,pred,zero_division=0),
        "flagged_rate":pred.mean()
    })
plt.tight_layout(); plt.savefig("day4_threshold_confusion_matrices.png",dpi=140); plt.show()
display(pd.DataFrame(kpi_rows))


## Task 5 — Final Evaluation & Save Artifact

The following metrics are calculated once on the **untouched 20% hold-out test set** using the calibrated model and the threshold selected only from development data.

In [ ]:
# Final test metrics
final_pred = (p_test >= best_threshold).astype(int)
final_metrics = pd.DataFrame([{
    "accuracy": accuracy_score(y_test,final_pred),
    "precision": precision_score(y_test,final_pred),
    "recall": recall_score(y_test,final_pred),
    "f1": f1_score(y_test,final_pred),
    "roc_auc": roc_auc_score(y_test,p_test),
    "average_precision": average_precision_score(y_test,p_test),
    "brier_score": brier_score_loss(y_test,p_test),
    "threshold": best_threshold
}])
display(final_metrics.T.rename(columns={0:"value"}))


In [ ]:
# ROC, PR and final confusion matrix
fpr,tpr,_=roc_curve(y_test,p_test)
plt.figure(figsize=(6,5)); plt.plot(fpr,tpr,label=f"AUC={final_metrics.loc[0,'roc_auc']:.3f}")
plt.plot([0,1],[0,1],"--"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Final Test"); plt.legend(); plt.tight_layout()
plt.savefig("day4_final_roc.png",dpi=140); plt.show()

prec,rec,_=precision_recall_curve(y_test,p_test)
plt.figure(figsize=(6,5)); plt.plot(rec,prec,label=f"AP={final_metrics.loc[0,'average_precision']:.3f}")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall Curve — Final Test")
plt.legend(); plt.tight_layout(); plt.savefig("day4_final_pr.png",dpi=140); plt.show()

ConfusionMatrixDisplay.from_predictions(y_test,final_pred,display_labels=["<=50K",">50K"],colorbar=False)
plt.title(f"Final Confusion Matrix — threshold={best_threshold:.2f}")
plt.tight_layout(); plt.savefig("day4_final_confusion_matrix.png",dpi=140); plt.show()


In [ ]:
# Save the complete calibrated pipeline
ARTIFACT = "final_adult_income_tuned_calibrated.joblib"
joblib.dump(calibrated_model, ARTIFACT)
print("Saved:", ARTIFACT)
print("Artifact contains preprocessing + estimator + probability calibration.")


## How to Infer on New Data

The saved artifact accepts a DataFrame containing the original 14 Adult feature columns. The production decision rule is the selected probability threshold rather than the default 0.50.

```python
import joblib
model = joblib.load("final_adult_income_tuned_calibrated.joblib")

proba = model.predict_proba(new_data)[:, 1]
prediction = (proba >= 0.42).astype(int)
```

**Production behavior:** keep the preprocessing and model together, preserve the same feature names/meaning, monitor missingness and class/base-rate drift, and periodically re-check calibration and the business threshold.

## Tuning Summary

- **Primary metric:** Precision.
- **Validation design:** stratified 5-fold CV during searches; fixed `random_state=42`.
- **Candidates:** Logistic Regression, Random Forest, HistGradientBoosting.
- **Final candidate:** HistGradientBoosting based on cross-validation/development performance.
- **Regularization:** Logistic `C`; HGB `l2_regularization`; RF depth/leaf size.
- **Calibration:** sigmoid/Platt scaling.
- **Threshold:** selected on development set by maximum F1.
- **Final evaluation:** one-time evaluation on untouched 20% test set.
- **Artifact:** `final_adult_income_tuned_calibrated.joblib`.

Run all cells after placing `adult_dataset.csv` beside the notebook to regenerate every metric, curve and artifact from scratch.